# SMAX checkpoint evaluation

This notebook evaluates saved SMAX checkpoints with deterministic rollouts and reports paper-table win rates at the training-seed level.


In [ ]:
import importlib
import os
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display


def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent):
        if (candidate / "main.py").is_file() and (candidate / "figures").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the pci_marl repo root from the current notebook working directory.")


ROOT = resolve_repo_root()
FIGURES_DIR = ROOT / "figures"
for path in (ROOT, FIGURES_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

command_buffer_flag = "--xla_gpu_enable_command_buffer="
existing_xla_flags = os.environ.get("XLA_FLAGS", "")
if command_buffer_flag not in existing_xla_flags:
    os.environ["XLA_FLAGS"] = f"{existing_xla_flags} {command_buffer_flag}".strip()
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import smax_checkpoint_eval_utils as scu
scu = importlib.reload(scu)

bootstrap_project = scu.bootstrap_project
discover_runs = scu.discover_runs
evaluate_map = scu.evaluate_map
evaluate_map_constraint = scu.evaluate_map_constraint
format_cross_map_latex = scu.format_cross_map_latex
ordered_map_names = scu.ordered_map_names
ordered_model_labels = scu.ordered_model_labels
save_summary_tables = scu.save_summary_tables

DEVICE = "cuda"
CHECKPOINT_CHOICE = "final"  # or "best"
DUPLICATE_POLICY = "latest"  # curated checkpoints_for_eval notebook policy; utility default remains "error"
EVAL_SEEDS = list(range(10001, 10065))
EVAL_EPISODES = 1
EVAL_SEED_BATCH_SIZE = 16
MAP_RESULTS = {}

bootstrap_project(DEVICE)
runs_df = discover_runs(ROOT / "checkpoints_for_eval", project_filter=None)
if runs_df.empty:
    raise RuntimeError("No runs were discovered under checkpoints_for_eval")

Repo root: /home/chamjin/research/pci_marl
Loaded 191 evaluation runs from checkpoints_for_eval
Maps: ['smacv2_10_units']
Models: ['pci-ippo', 'ippo', 'pci-mappo', 'mappo']


model_label                       ippo  mappo  pci-ippo  pci-mappo
map_name        constraint_label                                  
2s3z            tar_w=3              3      3         3          3
                tar_w=4              4      3         3          3
                vanilla              3      3         3          3
3m              tar_w=2              3      3         3          3
                vanilla              3      3         3          3
3s5z            tar_w=6              3      3         6          6
                vanilla              3      3         6          6
6h_vs_8z        tar_w=6              3      3         3          3
                vanilla              3      3         3          3
smacv2_10_units tar_w=6              3      3         3          3
                tar_w=8              3      3         3          3
                vanilla              3      2         3          3
smacv2_5_units  tar_w=3              3      3         3          3
                tar_w=4              3      3         3          3
                vanilla              2      3         3          3

Cached evaluations are reused automatically when checkpoint path, mtime, eval seed, and episode count match.


In [ ]:
# Select Maps and Models

TARGET_MAPS = ordered_map_names(runs_df["map_name"].dropna().unique().tolist()) # auto
# TARGET_MAPS = ['smacv2_5_units', 'smacv2_10_units'] # we can select map here!

TARGET_MODELS = ordered_model_labels(runs_df["model_label"].dropna().unique().tolist())

print(f"Repo root: {ROOT}")
print(f"Loaded {len(runs_df)} evaluation runs from checkpoints_for_eval")
print(f"Maps: {TARGET_MAPS}")
print(f"Models: {TARGET_MODELS}")
display(runs_df.groupby(["map_name", "constraint_label", "model_label"]).size().unstack(fill_value=0))
print("Cached evaluations are reused automatically when checkpoint path, mtime, eval seed, and episode count match.")

In [5]:
SMOKE_EVAL_SEEDS = EVAL_SEEDS[:2]

smoke_row = runs_df.sort_values(["map_name", "constraint_key", "model_label", "seed", "run_dir"]).iloc[0]
SMOKE_MAP = smoke_row["map_name"]
SMOKE_CONSTRAINT = smoke_row["constraint_key"]
SMOKE_MODEL = smoke_row["model_label"]
SMOKE_SEED = int(smoke_row["seed"])

smoke_runs = runs_df[
    (runs_df["map_name"] == SMOKE_MAP)
    & (runs_df["constraint_key"] == SMOKE_CONSTRAINT)
    & (runs_df["model_label"] == SMOKE_MODEL)
    & (runs_df["seed"] == SMOKE_SEED)
].copy()

display(smoke_runs[["map_name", "constraint_key", "model_label", "seed", "wandb_run_name", "run_dir"]])

smoke_result = evaluate_map_constraint(
    smoke_runs,
    SMOKE_MAP,
    SMOKE_CONSTRAINT,
    checkpoint_choice=CHECKPOINT_CHOICE,
    device=DEVICE,
    eval_episodes=EVAL_EPISODES,
    eval_seeds=SMOKE_EVAL_SEEDS,
    eval_seed_batch_size=min(EVAL_SEED_BATCH_SIZE, len(SMOKE_EVAL_SEEDS)),
    duplicate_policy=DUPLICATE_POLICY,
)
display(smoke_result["run_results"])
display(smoke_result["seed_summary"])
display(smoke_result["model_summary"])


,map_name,constraint_key,model_label,seed,wandb_run_name,run_dir
0,2s3z,tar_w=3,ippo,1,smax_2s3z_tar_w=3,/home/chamjin/research/pci_marl/checkpoints_fo...


/tmp/ipykernel_1345252/3041507688.py:18: UserWarning: std_success_rate is defined over training seeds. Filled 0.0 for rows with n_seeds=1.
map_name constraint_key model_label  n_seeds
    2s3z        tar_w=3        ippo        1
  smoke_result = evaluate_map_constraint(


,algorithm_type,checkpoint_choice,checkpoint_path,constraint_key,constraint_label,eval_max_reward,eval_mean_length,eval_mean_reward,eval_min_reward,eval_seed,...,map_name,metadata_path,model_key,model_label,num_episodes,project,run_dir,seed,train_seed,wandb_run_name
0,jax_ippo,final,/home/chamjin/research/pci_marl/checkpoints_fo...,tar_w=3,tar_w=3,0.859417,73.0,0.859417,0.859417,10001,...,2s3z,/home/chamjin/research/pci_marl/checkpoints_fo...,ippo,ippo,1,pci-marl-smax-jax-v3,/home/chamjin/research/pci_marl/checkpoints_fo...,1,1,smax_2s3z_tar_w=3
1,jax_ippo,final,/home/chamjin/research/pci_marl/checkpoints_fo...,tar_w=3,tar_w=3,2.000000,78.0,2.000000,2.000000,10002,...,2s3z,/home/chamjin/research/pci_marl/checkpoints_fo...,ippo,ippo,1,pci-marl-smax-jax-v3,/home/chamjin/research/pci_marl/checkpoints_fo...,1,1,smax_2s3z_tar_w=3


,map_name,constraint_key,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs,n_eval_seeds
0,2s3z,tar_w=3,tar_w=3,ippo,1,0.5,1.429708,75.5,1,2


,map_name,constraint_key,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs,n_eval_seeds
0,2s3z,tar_w=3,tar_w=3,ippo,0.5,0.0,1.429708,NaN,75.5,NaN,1,1,2


## Full evaluation

This cell evaluates each target map with a shared eval-seed set and preserves the aggregation shape:

- `run_results`: each train seed x eval seed result
- `seed_summary`: each train seed averaged over eval seeds
- `model_summary`: mean $\pm$ std over training seeds


In [12]:
MAP_RESULTS = {}
for map_name in TARGET_MAPS:
    result = evaluate_map(
        runs_df,
        map_name,
        checkpoint_choice=CHECKPOINT_CHOICE,
        device=DEVICE,
        eval_episodes=EVAL_EPISODES,
        eval_seeds=EVAL_SEEDS,
        eval_seed_batch_size=EVAL_SEED_BATCH_SIZE,
        model_order=TARGET_MODELS,
        duplicate_policy=DUPLICATE_POLICY,
    )
    if result["model_summary"].empty:
        print(f"Skipping {map_name}: no discovered runs")
        continue
    MAP_RESULTS[map_name] = result
    print(f"\n=== {map_name} ===")
    display(result["seed_summary"])
    display(result["model_summary"])
    print(result["latex"])


[SMAXWrapper] 시나리오 'smacv2_10_units': 10 allies vs 10 enemies
[SMAXBatchWrapper] 'smacv2_10_units': 10v10, n_envs=16
[SMAXBatchWrapper] obs_dim=257, action_dim=15
[SMAXBatchWrapper] 10v10 × 16 환경 (vmap 병렬)
[SMAXBatchWrapper] 배치 JIT warm-up 생략
[INFO] add_agent_id enabled: base_obs_dim=257, effective_obs_dim=267
Environment name: SMAXBatchWrapper
use_delta_critic: False
[SMAXWrapper] 시나리오 'smacv2_10_units': 10 allies vs 10 enemies
[SMAXBatchWrapper] 'smacv2_10_units': 10v10, n_envs=16
[SMAXBatchWrapper] allowed_targets (full source-level damage mask, window=6): {0: [0, 1, 2, 3, 4, 5], 1: [1, 2, 3, 4, 5, 6], 2: [2, 3, 4, 5, 6, 7], 3: [3, 4, 5, 6, 7, 8], 4: [4, 5, 6, 7, 8, 9], 5: [5, 6, 7, 8, 9, 0], 6: [6, 7, 8, 9, 0, 1], 7: [7, 8, 9, 0, 1, 2], 8: [8, 9, 0, 1, 2, 3], 9: [9, 0, 1, 2, 3, 4]}
[SMAXBatchWrapper] obs_dim=257, action_dim=15
[SMAXBatchWrapper] 10v10 × 16 환경 (vmap 병렬)
[SMAXBatchWrapper] 배치 JIT warm-up 생략
[INFO] add_agent_id enabled: base_obs_dim=257, effective_obs_dim=267
Environm

,map_name,constraint_key,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs,n_eval_seeds
0,smacv2_10_units,vanilla,vanilla,pci-ippo,1,0.359375,1.147378,49.718750,1,64
1,smacv2_10_units,vanilla,vanilla,pci-ippo,2,0.531250,1.348368,41.406250,1,64
2,smacv2_10_units,vanilla,vanilla,pci-ippo,42,0.343750,1.103817,51.062500,1,64
3,smacv2_10_units,vanilla,vanilla,ippo,1,0.046875,0.625026,47.140625,1,64
4,smacv2_10_units,vanilla,vanilla,ippo,2,0.125000,0.743469,49.562500,1,64
5,smacv2_10_units,vanilla,vanilla,ippo,42,0.093750,0.737782,54.812500,1,64
6,smacv2_10_units,vanilla,vanilla,pci-mappo,1,0.515625,1.335898,48.109375,1,64
7,smacv2_10_units,vanilla,vanilla,pci-mappo,2,0.515625,1.325150,44.703125,1,64
8,smacv2_10_units,vanilla,vanilla,pci-mappo,42,0.625000,1.479402,45.921875,1,64
9,smacv2_10_units,vanilla,vanilla,mappo,1,0.421875,1.257692,42.859375,1,64


,map_name,constraint_key,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs,n_eval_seeds
0,smacv2_10_units,vanilla,vanilla,pci-ippo,0.411458,0.104036,1.199855,0.130448,47.395833,5.230463,3,3,64
1,smacv2_10_units,vanilla,vanilla,ippo,0.088542,0.039322,0.702092,0.066802,50.505208,3.921854,3,3,64
2,smacv2_10_units,vanilla,vanilla,pci-mappo,0.552083,0.063148,1.380150,0.086123,46.244792,1.725932,3,3,64
3,smacv2_10_units,vanilla,vanilla,mappo,0.468750,0.066291,1.310652,0.074898,43.789062,1.314777,2,2,64
4,smacv2_10_units,tar_w=6,tar_w=6,pci-ippo,0.062500,0.031250,0.711580,0.066261,46.296875,3.516875,3,3,64
5,smacv2_10_units,tar_w=6,tar_w=6,ippo,0.005208,0.009021,0.512939,0.037302,39.958333,3.972880,3,3,64
6,smacv2_10_units,tar_w=6,tar_w=6,pci-mappo,0.177083,0.032526,0.916583,0.061550,44.010417,2.395590,3,3,64
7,smacv2_10_units,tar_w=6,tar_w=6,mappo,0.083333,0.023868,0.783284,0.037856,45.567708,4.014571,3,3,64
8,smacv2_10_units,tar_w=8,tar_w=8,pci-ippo,0.135417,0.126295,0.844225,0.175004,48.536458,3.071151,3,3,64
9,smacv2_10_units,tar_w=8,tar_w=8,ippo,0.010417,0.009021,0.505757,0.023394,39.937500,4.086946,3,3,64


\begin{table}[t]
\centering
\caption{Deterministic SMAX win rate on smacv2_10_units using final checkpoints. Rows are grouped by training constraint. Mean $\pm$ std are computed over seeds, not episodes.}}
\label{tab:smax_smacv2_10_units_final}
\begin{tabular}{lcccc}
\toprule
Constraint & pci-ippo & ippo & pci-mappo & mappo \\
\midrule
vanilla & 41.1 \scriptsize $\pm$ 10.4 & 8.9 \scriptsize $\pm$ 3.9 & 55.2 \scriptsize $\pm$ 6.3 & 46.9 \scriptsize $\pm$ 6.6 \\
tar_w=6 & 6.2 \scriptsize $\pm$ 3.1 & 0.5 \scriptsize $\pm$ 0.9 & 17.7 \scriptsize $\pm$ 3.3 & 8.3 \scriptsize $\pm$ 2.4 \\
tar_w=8 & 13.5 \scriptsize $\pm$ 12.6 & 1.0 \scriptsize $\pm$ 0.9 & 37.5 \scriptsize $\pm$ 8.7 & 20.3 \scriptsize $\pm$ 12.4 \\
\bottomrule
\end{tabular}
\end{table}


In [7]:
for map_name, result in MAP_RESULTS.items():
    run_df = result["run_results"].copy()
    seed_summary = result["seed_summary"].copy()
    model_summary = result["model_summary"].copy()

    eval_seed_sets = (
        run_df.groupby(["constraint_key", "model_label", "train_seed"])["eval_seed"]
        .agg(lambda values: tuple(sorted(set(int(v) for v in values))))
        .reset_index(name="eval_seed_set")
    )
    method_seed_counts = eval_seed_sets.groupby(["constraint_key", "model_label"])["eval_seed_set"].nunique()
    assert (method_seed_counts == 1).all(), f"Inconsistent eval_seed sets detected for {map_name}:\n{method_seed_counts}"

    assert not (run_df["train_seed"] == run_df["eval_seed"]).any(), f"train_seed and eval_seed collided for {map_name}"

    expected_seed_rows = (
        run_df.groupby(["constraint_key", "model_label"])["train_seed"]
        .nunique()
        .reset_index(name="expected_rows")
    )
    actual_seed_rows = (
        seed_summary.groupby(["constraint_key", "model_label"])
        .size()
        .reset_index(name="actual_rows")
    )
    row_check = expected_seed_rows.merge(actual_seed_rows, on=["constraint_key", "model_label"], how="left")
    assert (row_check["expected_rows"] == row_check["actual_rows"]).all(), f"seed_summary row mismatch for {map_name}:\n{row_check}"

    expected_std = (
        seed_summary.groupby(["constraint_key", "model_label"])["eval_success_rate"]
        .std()
        .reset_index(name="expected_std_success_rate")
    )
    std_check = model_summary.merge(expected_std, on=["constraint_key", "model_label"], how="left")
    observed = std_check["std_success_rate"].fillna(0.0).to_numpy()
    expected = std_check["expected_std_success_rate"].fillna(0.0).to_numpy()
    assert ((observed - expected).__abs__() < 1.0e-12).all(), f"std_success_rate mismatch for {map_name}:\n{std_check}"

    if DUPLICATE_POLICY != "all":
        assert (seed_summary["n_runs"] == 1).all(), f"duplicate runs leaked into seed_summary for {map_name}:\n{seed_summary[['constraint_key', 'model_label', 'seed', 'n_runs']]}"

print("Protocol assertions passed.")


Protocol assertions passed.


In [8]:
INSPECT_MAP = TARGET_MAPS[0]
if INSPECT_MAP not in MAP_RESULTS:
    raise RuntimeError(f"Run the full evaluation cell first. Missing {INSPECT_MAP}.")

display(MAP_RESULTS[INSPECT_MAP]["seed_summary"])
display(MAP_RESULTS[INSPECT_MAP]["model_summary"])
print(MAP_RESULTS[INSPECT_MAP]["latex"])


,map_name,constraint_key,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs,n_eval_seeds
0,3m,vanilla,vanilla,pci-ippo,1,0.968750,1.960417,23.265625,1,64
1,3m,vanilla,vanilla,pci-ippo,2,0.984375,1.982292,24.531250,1,64
2,3m,vanilla,vanilla,pci-ippo,42,0.984375,1.979167,22.156250,1,64
3,3m,vanilla,vanilla,ippo,1,0.796875,1.764583,16.375000,1,64
4,3m,vanilla,vanilla,ippo,2,0.828125,1.802083,16.890625,1,64
5,3m,vanilla,vanilla,ippo,42,0.828125,1.789583,14.953125,1,64
6,3m,vanilla,vanilla,pci-mappo,1,0.906250,1.880208,27.984375,1,64
7,3m,vanilla,vanilla,pci-mappo,2,0.984375,1.981250,24.343750,1,64
8,3m,vanilla,vanilla,pci-mappo,42,0.906250,1.876042,27.265625,1,64
9,3m,vanilla,vanilla,mappo,1,0.906250,1.873958,18.265625,1,64


,map_name,constraint_key,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs,n_eval_seeds
0,3m,vanilla,vanilla,pci-ippo,0.979167,0.009021,1.973958,0.011831,23.317708,1.188356,3,3,64
1,3m,vanilla,vanilla,ippo,0.817708,0.018042,1.785417,0.019094,16.072917,1.003453,3,3,64
2,3m,vanilla,vanilla,pci-mappo,0.932292,0.045105,1.912500,0.059576,26.531250,1.928216,3,3,64
3,3m,vanilla,vanilla,mappo,0.901042,0.023868,1.880556,0.027170,16.703125,1.790069,3,3,64
4,3m,tar_w=2,tar_w=2,pci-ippo,0.640625,0.315996,1.534375,0.449719,33.718750,2.273800,3,3,64
5,3m,tar_w=2,tar_w=2,ippo,0.375000,0.015625,1.224306,0.007818,15.994792,0.306319,3,3,64
6,3m,tar_w=2,tar_w=2,pci-mappo,0.906250,0.054127,1.890972,0.061098,34.088542,0.779738,3,3,64
7,3m,tar_w=2,tar_w=2,mappo,0.505208,0.077076,1.400347,0.086992,15.729167,0.203925,3,3,64


\begin{table}[t]
\centering
\caption{Deterministic SMAX win rate on 3m using final checkpoints. Rows are grouped by training constraint. Mean $\pm$ std are computed over seeds, not episodes.}}
\label{tab:smax_3m_final}
\begin{tabular}{lcccc}
\toprule
Constraint & pci-ippo & ippo & pci-mappo & mappo \\
\midrule
vanilla & 97.9 \scriptsize $\pm$ 0.9 & 81.8 \scriptsize $\pm$ 1.8 & 93.2 \scriptsize $\pm$ 4.5 & 90.1 \scriptsize $\pm$ 2.4 \\
tar_w=2 & 64.1 \scriptsize $\pm$ 31.6 & 37.5 \scriptsize $\pm$ 1.6 & 90.6 \scriptsize $\pm$ 5.4 & 50.5 \scriptsize $\pm$ 7.7 \\
\bottomrule
\end{tabular}
\end{table}


In [9]:
if not MAP_RESULTS:
    raise RuntimeError("Run the full evaluation cell first.")

combined_model_summary = pd.concat(
    [MAP_RESULTS[map_name]["model_summary"] for map_name in TARGET_MAPS if map_name in MAP_RESULTS],
    ignore_index=True,
)
display(combined_model_summary)

latex_code = format_cross_map_latex(
    combined_model_summary,
    checkpoint_choice=CHECKPOINT_CHOICE,
    map_order=TARGET_MAPS,
    model_order=TARGET_MODELS,
)
print(latex_code)

paths = save_summary_tables(
    combined_model_summary,
    FIGURES_DIR,
    checkpoint_choice=CHECKPOINT_CHOICE,
    map_order=TARGET_MAPS,
    model_order=TARGET_MODELS,
)
print(f"Saved CSV: {paths['csv']}")
print(f"Saved TeX: {paths['tex']}")


,map_name,constraint_key,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs,n_eval_seeds
0,3m,vanilla,vanilla,pci-ippo,0.979167,0.009021,1.973958,1.183106e-02,23.317708,1.188356,3,3,64
1,3m,vanilla,vanilla,ippo,0.817708,0.018042,1.785417,1.909407e-02,16.072917,1.003453,3,3,64
2,3m,vanilla,vanilla,pci-mappo,0.932292,0.045105,1.912500,5.957568e-02,26.531250,1.928216,3,3,64
3,3m,vanilla,vanilla,mappo,0.901042,0.023868,1.880556,2.717000e-02,16.703125,1.790069,3,3,64
4,3m,tar_w=2,tar_w=2,pci-ippo,0.640625,0.315996,1.534375,4.497190e-01,33.718750,2.273800,3,3,64
5,3m,tar_w=2,tar_w=2,ippo,0.375000,0.015625,1.224306,7.818285e-03,15.994792,0.306319,3,3,64
6,3m,tar_w=2,tar_w=2,pci-mappo,0.906250,0.054127,1.890972,6.109829e-02,34.088542,0.779738,3,3,64
7,3m,tar_w=2,tar_w=2,mappo,0.505208,0.077076,1.400347,8.699216e-02,15.729167,0.203925,3,3,64
8,2s3z,vanilla,vanilla,pci-ippo,0.906250,0.041340,1.888257,4.874379e-02,75.114583,7.887922,3,3,64
9,2s3z,vanilla,vanilla,ippo,0.994792,0.009021,1.994368,9.754118e-03,55.578125,5.775905,3,3,64


\begin{table*}[t]
\centering
\caption{Deterministic SMAX win rate using final checkpoints. Rows are grouped by map and training constraint. Mean $\pm$ std are computed over seeds, not episodes.}}
\label{tab:smax_cross_map_final}
\resizebox{\linewidth}{!}{%
\begin{tabular}{llcccc}
\toprule
Map & Constraint & pci-ippo & ippo & pci-mappo & mappo \\
\midrule
3m & vanilla & 97.9 \scriptsize$\pm$ 0.9 & 81.8 \scriptsize$\pm$ 1.8 & 93.2 \scriptsize$\pm$ 4.5 & 90.1 \scriptsize$\pm$ 2.4 \\
3m & tar_w=2 & 64.1 \scriptsize$\pm$ 31.6 & 37.5 \scriptsize$\pm$ 1.6 & 90.6 \scriptsize$\pm$ 5.4 & 50.5 \scriptsize$\pm$ 7.7 \\
2s3z & vanilla & 90.6 \scriptsize$\pm$ 4.1 & 99.5 \scriptsize$\pm$ 0.9 & 97.4 \scriptsize$\pm$ 1.8 & 99.0 \scriptsize$\pm$ 0.9 \\
2s3z & tar_w=3 & 91.7 \scriptsize$\pm$ 2.4 & 51.6 \scriptsize$\pm$ 45.2 & 69.8 \scriptsize$\pm$ 33.7 & 68.2 \scriptsize$\pm$ 5.9 \\
2s3z & tar_w=4 & 66.1 \scriptsize$\pm$ 30.0 & 94.3 \scriptsize$\pm$ 4.8 & 59.4 \scriptsize$\pm$ 22.0 & 96.9 \scriptsize$\pm$